# 4.3 🔐 Providers Productivos: Autenticación, Retries y Buenas Prácticas

En la sección anterior implementamos un **MarketDataProvider** desacoplado.

Pero en el mundo real, los proveedores de datos no son abiertos como Yahoo Finance.

La mayoría requieren:

- 🔑 API Keys
- 🔐 Tokens Bearer
- 🪪 OAuth2
- 🔄 Retries
- ⚡ Caching

En esta sección veremos cómo llevar nuestros providers a un nivel **productivo**.


## 🧭 Arquitectura extendida del Provider

```{mermaid}
flowchart LR
    A[Modelos] --> B[Provider Interface]
    B --> C[Auth Layer]
    C --> D[Retry Layer]
    D --> E[Cache Layer]
    E --> F[External API]
```


## 🔑 Autenticación con API Keys

In [ ]:
import requests

API_KEY = "TU_API_KEY"

def obtener_precio_alpha_vantage(ticker: str):
    url = "https://www.alphavantage.co/query"

    params = {
        "function": "GLOBAL_QUOTE",
        "symbol": ticker,
        "apikey": API_KEY,
    }

    response = requests.get(url, params=params, timeout=10)

    return response.status_code, response.json()


## 🔐 Autenticación con Bearer Tokens

In [ ]:
import requests

TOKEN = "TU_TOKEN"

def consultar_api_protegida(endpoint: str):

    headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Accept": "application/json",
    }

    response = requests.get(endpoint, headers=headers, timeout=10)

    return response.status_code, response.json()


## 🪪 OAuth2 (flujo conceptual)

```{mermaid}
sequenceDiagram
    participant Cliente
    participant AuthServer
    participant API

    Cliente->>AuthServer: Solicita Token
    AuthServer-->>Cliente: Access Token
    Cliente->>API: Request + Token
    API-->>Cliente: Response
```


## 🔄 Retries y resiliencia

In [ ]:
import requests
import time

def request_con_retry(url: str, retries: int = 3):

    for intento in range(retries):

        try:
            r = requests.get(url, timeout=5)

            if r.status_code == 200:
                return r.json()

        except requests.exceptions.RequestException:
            pass

        time.sleep(2)

    raise ConnectionError("API no disponible")


```{mermaid}
flowchart TD
    A[Request] --> B{200 OK?}
    B -->|Sí| C[Return Data]
    B -->|No| D[Retry]
    D --> E{Max?}
    E -->|No| A
    E -->|Sí| F[Error]
```


## ⚡ Caching

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def obtener_precio_cacheado(ticker: str):

    print("Consultando API...")
    return 100.0

print(obtener_precio_cacheado("AAPL"))
print(obtener_precio_cacheado("AAPL"))


## 🎯 Cierre

Un provider productivo gestiona:

- Seguridad
- Disponibilidad
- Costos
- Performance

Esto separa scripts exploratorios de plataformas analíticas empresariales 🚀
